# 석천사거리(204820) 중심 DQN 실험 v3 — Colab

위에서부터 **▶ 버튼**을 차례로 누르면 됩니다. Colab Pro+의 백그라운드 실행을 켜 두면 브라우저를 닫아도 계속 돕니다.

- 차량: 석천사거리에서 기록된 회전 1건 = 시뮬레이션 차량 1대 (2022년 19일). 이동별 15분 교통량을 15분 동안 무작위 시각에 도착시킨다.
- 신호: AI Hub 실제 운영 신호(한 현시 밀림 보정, 주 기준선) / 지도 내장 신호
- DQN 운영 제약: 보행 최소 녹색(길주로 횡단 46초, 석천로 횡단 33초), 보행 최대 대기(155초, 120초), 최대 녹색 120초, 정지선 차량 120초 대기 보장
- 날짜 분할: `config/focused_split.json` (학습 13일 · 검증 2일 · 시험 4일)
- 결과는 Google Drive의 `내 드라이브/FutureTraffic/focused_results/<실험이름>/`에 저장됩니다.

## 1. Google Drive 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 실행 묶음 풀기
`내 드라이브/FutureTraffic/focused_package.zip`을 Colab 작업 폴더로 풉니다.

In [ ]:
from pathlib import Path
import zipfile, shutil
DRIVE = Path('/content/drive/MyDrive/FutureTraffic')
PACKAGE = DRIVE / 'focused_package.zip'
REPO = Path('/content/FutureTraffic')
assert PACKAGE.exists(), f'묶음 파일이 없습니다: {PACKAGE}'
if REPO.exists():
    shutil.rmtree(REPO)
with zipfile.ZipFile(PACKAGE) as z:
    z.extractall(REPO)
print('풀기 완료:', sorted(p.name for p in REPO.iterdir()))

## 3. 프로그램 설치 (처음 몇 분)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements.txt')], check=True)
print('설치 완료')

## 4. 기본 검사
차량 수·경로·신호 틀이 맞는지 확인합니다. `OK`가 나오면 정상입니다.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'tests/test_focused.py', 'tests/test_constraints.py', '-v'],
               cwd=REPO, check=True)
import os
CPUS = os.cpu_count()
print('CPU 코어 수:', CPUS)

## 5. 실험 설정
- `EXPERIMENT`: 결과 폴더 이름. 설정을 바꿀 때마다 이름도 바꾸세요.
- `MIN_GREEN`: 보행자가 건너지 않는 현시의 최소 녹색(초). 보행 현시는 자동으로 46초·33초가 적용됩니다.
- `MAX_WAIT`: 정지선 차량 최대 대기 보장(초).
- 황색은 자동입니다(실제 신호 4초, 지도 신호 3초).
- `TIMESTEPS`: 모델 하나의 학습 단계 수. 로컬 실험은 100,000단계 × seed 3개였습니다.

In [ ]:
EXPERIMENT = 'final_v3'       # @param {type:"string"}
MIN_GREEN = 15                # @param {type:"integer"}
MAX_WAIT = 120                # @param {type:"integer"}
MAX_GREEN = 120               # @param {type:"integer"}
TIMESTEPS = 300000            # @param {type:"integer"}
SEEDS = [42, 43, 44, 45, 46]
VARIANTS = ['embedded', 'aihub']
RESULTS_LOCAL = Path('/content/results') / EXPERIMENT
RESULTS_DRIVE = DRIVE / 'focused_results' / EXPERIMENT
RESULTS_LOCAL.mkdir(parents=True, exist_ok=True)
RESULTS_DRIVE.mkdir(parents=True, exist_ok=True)
import json
SPLIT = json.loads((REPO / 'config/focused_split.json').read_text())
print('학습 날짜', len(SPLIT['train']), '시험 날짜', SPLIT['test'])

## 6. DQN 학습 (코어 수만큼 동시에)
이미 끝난 모델은 건너뜁니다. 진행 상황이 5분마다 출력됩니다.

In [ ]:
import time, itertools
def sync_to_drive():
    """새로 생기거나 바뀐 결과 파일만 Drive로 복사한다(차량 파일 routes는 제외)."""
    for path in RESULTS_LOCAL.rglob('*'):
        if path.is_dir() or 'routes' in path.parts:
            continue
        target = RESULTS_DRIVE / path.relative_to(RESULTS_LOCAL)
        if not target.exists() or target.stat().st_mtime < path.stat().st_mtime:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)

jobs = []
for variant, seed in itertools.product(VARIANTS, SEEDS):
    done_drive = RESULTS_DRIVE / f'dqn_{variant}' / f'seed_{seed}' / 'dqn_204820.zip'
    if done_drive.exists():
        target = RESULTS_LOCAL / f'dqn_{variant}' / f'seed_{seed}'
        target.mkdir(parents=True, exist_ok=True)
        shutil.copytree(done_drive.parent, target, dirs_exist_ok=True)
        continue
    jobs.append((variant, seed))
print('학습할 모델', jobs)
env = dict(os.environ, OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', PYTHONUNBUFFERED='1')
running = []
pending = list(jobs)
last = 0
while pending or running:
    while pending and len(running) < max(1, CPUS):
        variant, seed = pending.pop(0)
        log = open(RESULTS_LOCAL / f'train_{variant}_{seed}.log', 'w')
        cmd = [sys.executable, 'src/focused_experiment.py', 'train', '--variant', variant, '--seed', str(seed),
               '--timesteps', str(TIMESTEPS), '--split-file', 'config/focused_split.json',
               '--min-green', str(MIN_GREEN), '--max-green', str(MAX_GREEN),
               '--max-wait', str(MAX_WAIT),
               '--checkpoint-freq', '50000', '--out', str(RESULTS_LOCAL)]
        running.append((variant, seed, subprocess.Popen(cmd, cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT)))
    for item in list(running):
        if item[2].poll() is not None:
            running.remove(item)
            print(f'끝남: {item[0]} seed {item[1]} (종료 코드 {item[2].returncode})')
            sync_to_drive()
    if time.time() - last > 300:
        last = time.time()
        for variant, seed, _ in running:
            mon = RESULTS_LOCAL / f'dqn_{variant}' / f'seed_{seed}' / 'monitor.csv'
            n = max(0, sum(1 for _ in open(mon)) - 2) if mon.exists() else 0
            print(f'  {variant} seed {seed}: 에피소드 {n} (약 {n*360:,}단계)')
        sync_to_drive()
    time.sleep(20)
sync_to_drive()
print('학습 완료')

## 7. 시험 날짜 평가와 보고서
모든 정책이 같은 차량 파일을 씁니다. 무작위 신호와 '자료 그대로' 신호는 첫 시험 날짜에서만 봅니다.

In [ ]:
MAIN = ['fixed_aihub_corrected', 'fixed_embedded', 'actuated_aihub', 'actuated_embedded', 'dqn_aihub', 'dqn_embedded']
for i, date in enumerate(SPLIT['test']):
    policies = MAIN + (['random_aihub', 'random_embedded', 'fixed_aihub_asis'] if i == 0 else [])
    subprocess.run([sys.executable, 'src/focused_experiment.py', 'evaluate', '--date', date, '--policies', *policies,
                    '--train-seeds', *map(str, SEEDS), '--workers', str(max(1, CPUS)),
                    '--out', str(RESULTS_LOCAL)], cwd=REPO, env=env, check=True)
    sync_to_drive()
subprocess.run([sys.executable, 'src/focused_experiment.py', 'report', '--dates', *SPLIT['test'],
                '--out', str(RESULTS_LOCAL)], cwd=REPO, env=env, check=True)
sync_to_drive()
print('평가 완료')

## 8. 보고서 보기

In [ ]:
from IPython.display import Markdown, display
display(Markdown((RESULTS_LOCAL / 'report.md').read_text(encoding='utf-8')))